In [ ]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import requests
import pandas as pd
import numpy as np
from time import sleep

# CONFIG 
OUTPUT_FILE = "domestic_commodity_standardized.csv"

RATE_CURRENCIES = {"BYR", "TMT"}


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
."Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.

return fill_usd_tmt(df)
elif currency_upper == "BYR":
return fill_usd_byr(df)
    else:
        return fill_usd_via_implicit_rate(df)


def recompute_price_per_unit(df: pd.DataFrame) -> pd.DataFrame:
    """Recompute price_per_unit for any row that now has price_usd."""
    mask = df["price_usd"].notna() & df["price_per_unit"].isna()
    if mask.any():
        df.loc[mask, "price_per_unit"] = df.loc[mask].apply(
            lambda r: compute_price_per_unit(r["price_usd"], r["unit"]), axis=1
        )
    return df


# SERIES BUILDER


FINAL_COLS = [
    "date", "price_usd", "price_local", "currency", "price_per_unit", "unit_std",
    "commodity_name", "iso3_country_code", "country", "market",
    "price_type", "unit", "price_source", "fill_method"
]


def build_series_df(datapoints, meta: dict, price_source: str) -> pd.DataFrame | None:
    rows = []
    for dp in datapoints:
        price_usd   = dp.get("price_value_dollar")
        price_local = dp.get("price_value")
        if price_local is None:
            price_local = dp.get("price_value_nominal")

        if price_usd is None and price_local is None:
            continue

        rows.append({
            "date":              dp["date"],
            "price_usd":         price_usd,
            "price_local":       price_local,
            "currency":          meta["currency"],
            "price_per_unit":    compute_price_per_unit(price_usd, meta["unit"]),
            "unit_std":          get_unit_std(meta["unit"]),
            "commodity_name":    meta["commodity_name"],
            "iso3_country_code": meta["iso3"],
            "country":           meta["country"],
            "market":            meta["market"],
            "price_type":        meta["price_type"],
            "unit":              meta["unit"],
            "price_source":      price_source,
            "fill_method":       "original",
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])

    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()

    df = (df.groupby("date", as_index=False)
            .apply(lambda g: g.loc[g.isna().sum(axis=1).idxmin()], include_groups=False)
            .reset_index(drop=True))

    full_range = pd.date_range(start=df["date"].min(),
                               end=df["date"].max(), freq="MS")
    df = (df.set_index("date")
            .reindex(full_range)
            .rename_axis("date")
            .reset_index())

    for col in ["commodity_name", "iso3_country_code", "country", "market",
                "price_type", "unit", "unit_std", "currency",
                "price_source", "fill_method"]:
        df[col] = df[col].ffill().bfill()

    df["fill_method"] = df["fill_method"].fillna("original")
    return df



# DOMESTIC SERIES
print("=" * 60)
print("FETCHING DOMESTIC SERIES")
print("=" * 60)

url_dom  = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieDomestic/?newerThan=12 months ago"
dom_data = requests.get(url_dom).json()
print(f"Total domestic series: {len(dom_data['results'])}")

all_rows = []

fill_stats = {
    "original":             0,
    "implicit_rate":        0,
    "hardcoded_rate_tmt":   0,
    "hardcoded_rate_byr":   0,
    "unfilled":             0,
}

for item in dom_data["results"]:
    uuid           = item["uuid"]
    commodity_name = item.get("commodity_name", "Unknown")
    country        = item.get("country_name", "Unknown")
    iso3           = item.get("iso3_country_code", "Unknown")
    market         = item.get("market_name", "Unknown")
    price_type     = item.get("price_type", "Unknown")
    unit           = item.get("measure_unit_label", "Unknown")
    currency       = item.get("currency", "Unknown")

    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()
    if resp["count"] == 0:
        sleep(0.2)
        continue

    datapoints = resp["results"][0]["datapoints"]
    meta = dict(commodity_name=commodity_name, iso3=iso3, country=country,
                market=market, price_type=price_type, unit=unit, currency=currency)

    df = build_series_df(datapoints, meta, price_source="Domestic")
    if df is None:
        sleep(0.2)
        continue

    if df["price_usd"].isna().all() and df["price_local"].isna().all():
        print(f"  [SKIP] All-null series: {commodity_name} | {market}, {country}")
        sleep(0.2)
        continue

    missing_before = df["price_usd"].isna().sum()
    if missing_before > 0:
        df, n_filled = fill_usd(df, currency)
    else:
        n_filled = 0

    df = recompute_price_per_unit(df)

    missing_after = df["price_usd"].isna().sum()
    fill_stats["unfilled"] += missing_after
    if n_filled > 0:
        method = (df.loc[df["fill_method"] != "original", "fill_method"]
                    .iloc[0]
                  if (df["fill_method"] != "original").any()
                  else "unknown")
        fill_stats[method] = fill_stats.get(method, 0) + n_filled
    else:
        fill_stats["original"] += (missing_before - missing_after)

    print(f"  {commodity_name} | {market}, {country} | "
          f"{len(df)} rows | filled {n_filled}/{missing_before} missing USD")

    all_rows.append(df[FINAL_COLS])
    sleep(0.2)


# SAVE

print("\n" + "=" * 60)
print("SAVING")
print("=" * 60)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    final_df = pd.concat(all_rows, ignore_index=True)

final_df = final_df.sort_values(
    ["country", "market", "commodity_name", "date"]
).reset_index(drop=True)

print(f"\nTotal rows          : {len(final_df):,}")
print(f"Missing price_usd   : {final_df['price_usd'].isna().sum():,}")
print(f"Missing price_local : {final_df['price_local'].isna().sum():,}")
print(f"\nFill method breakdown:")
print(final_df["fill_method"].value_counts().to_string())

print(f"\nFill strategy stats:")
for method, count in fill_stats.items():
    print(f"  {method}: {count:,}")

final_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved → {OUTPUT_FILE}")

# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

FETCHING DOMESTIC SERIES
Total domestic series: 3383
  Bread | Kabul, Afghanistan | 314 rows | filled 0/1 missing USD
  Bread | Herat, Afghanistan | 314 rows | filled 0/0 missing USD
  Wheat | Kandahar, Afghanistan | 314 rows | filled 0/3 missing USD
  Wheat (flour) | Herat, Afghanistan | 314 rows | filled 0/0 missing USD
  Wheat (flour) | Kandahar, Afghanistan | 314 rows | filled 0/3 missing USD
  Wheat (flour) | Jalalabad, Afghanistan | 314 rows | filled 0/0 missing USD
  Bread | Kandahar, Afghanistan | 314 rows | filled 0/3 missing USD
  Wheat | Jalalabad, Afghanistan | 314 rows | filled 0/0 missing USD
  Wheat | Herat, Afghanistan | 314 rows | filled 0/0 missing USD
  Bread | Jalalabad, Afghanistan | 314 rows | filled 0/0 missing USD
  Wheat (flour) | Kabul, Afghanistan | 314 rows | filled 0/0 missing USD
  Wheat | Kabul, Afghanistan | 314 rows | filled 0/0 missing USD
  Rice (long grain) | Lunda Norte, Angola | 60 rows | filled 0/0 missing USD
  Palm Oil | Cuando Cubango, Angola |